# OSRT v6b — GRPO on Colab (G4 / RTX PRO 6000 Blackwell, 96GB)

RL with verifiable maths rewards from the **SFT-v4 checkpoint soup**
(`acc_on 20.0% / acc_off 13.5%`, n=200, measured 2026-08-10).

**Secrets:** add `HF_TOKEN` and `WANDB_API_KEY` in the Colab key sidebar (🔑).

---
## Read this first: what is fixed, and what is not

**The objective is fixed.** Rollouts are sampled at `T=0.4`; the log-probs that
feed the gradient are now scored at the same temperature. They were not, and the
consequence was measured: the historical update sat **77–83° away** from the
corrected objective. So wave 2's null result was never a test of GRPO on this
model. Also landed: `kl_coeff` 0.15→0.04, `top_p` 0.95→1.0 (nucleus sampling
renormalises, so only 1.0 matches an untruncated scorer), and KL applied to every
non-empty rollout rather than only advantaged ones.

**The reward is NOT fixed.** `5.0 + 0.2 + 0.3 = 5.5` maximum, so **90.9%** rides
on the final number, and the `+0.3` bonus counts reasoning *steps* without
validating them. Observed directly: whole rollout groups scoring the full
**+5.50** on traces that computed 40 and answered 10, or that reached the right
answer via `17% of 170 = 25.7` (it is 28.9). So a long run against this reward
optimises a target we know is misaligned — see
`docs/specs/2026-08-10-verification-reward-prereg.md`.

**Latest result.** 40 corrected steps → `acc_on 16.5% / acc_off 15.0%`. Paired
against the soup: `+3.50pp, CI [-0.50,+7.50], p=0.105` — **crosses zero**. That
is *non-degradation*, and it is the first GRPO run here that did not lose
ground (wave 2 was `+6.50pp, CI [+1.50,+11.50], p=0.012`, which excluded zero).
It is **not** evidence of improvement.

---
## Metric discipline

| role | metric |
|---|---|
| **primary** | absolute `acc_on` vs the soup's 20.0% |
| co-primary | semantic verification accuracy *(not built yet)* |
| control | `acc_off` — retained capability, never a target |
| diagnostic | `delta` — must not collapse, must never be maximised |
| diagnostic | weight drift — **not** a verdict |

`delta` is demoted because it improves when `acc_off` **falls**: the wave-2 soup
gained 4pp of delta largely by losing 3pp of `acc_off`.

These 200 problems are the **development panel** — right for selection and
diagnostics, wrong for a final claim, because checkpoints and soups were chosen
after seeing them. Final claims need `problem_offset=200` (~1119 untouched
GSM8K problems) with candidates declared in advance.

In [ ]:
# ── 1. Repo sync + secrets + RUN KNOBS ───────────────────────────────
import os, sys

BRANCH = "feat/grpo-v6b"
REPO = "https://github.com/CodeHalwell/OSRT-605M-A269M.git"
if not os.path.isdir("/content/osrt"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/osrt
else:
    # ALWAYS SYNC. An earlier version only cloned when the directory was absent,
    # so re-running was a NO-OP and the repo silently stayed frozen at whatever
    # the first clone fetched — for the life of the VM, with no error shown.
    !cd /content/osrt && git remote set-url origin {REPO} \
        && git fetch -q --depth 1 origin {BRANCH} \
        && git checkout -q -B {BRANCH} FETCH_HEAD
!cd /content/osrt && git log --oneline -1 && git rev-parse --abbrev-ref HEAD

%pip -q install -U "transformers>=5.3.0" datasets tokenizers safetensors wandb huggingface_hub
sys.path.insert(0, "/content/osrt/src")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── KNOBS — one source of truth for the sanity cell AND the run ──────
TOTAL_STEPS = 40      # 900 for a full run, and NOT against the current reward
MAX_GEN_LEN = 1024    # length-ramp DENOMINATOR: free below 80% (819 tokens)

# LR probe (0 = use the config value). COMPARE AT STEP 30: lr_at_step computes
# warmup as peak_lr*eff/warmup_steps independently of total_steps, and at step
# 30 eff == warmup_steps so the cosine term is cos(0)=1 and returns exactly
# peak_lr. Steps 0-30 are schedule-matched for ANY total_steps; step 31 diverges.
# HOLD HRA_LR while raising PEAK_LR — measured HRA:base movement was 49x against
# a 10x lr ratio, so scaling both preserves the adapter dominance.
PEAK_LR  = 0.0        # e.g. 5e-6
HRA_LR   = 0.0        # e.g. 1.5e-5 to hold it -> 3x ratio
RUN_TAG  = ""         # e.g. "lr5e6" — REQUIRED for a sweep, see below
KL_ABORT = 0.0        # e.g. 0.25 (wave 1's worst was 0.33)

# RUN_TAG suffixes stage_prefix and the W&B name. Without it every setting
# writes the same <prefix>_step_N.pt, so HF pushes COLLIDE and a run can resume
# from a checkpoint trained at a different LR.
for k, v in dict(TOTAL_STEPS=TOTAL_STEPS, MAX_GEN_LEN=MAX_GEN_LEN,
                 PEAK_LR=PEAK_LR, HRA_LR=HRA_LR, RUN_TAG=RUN_TAG,
                 KL_ABORT=KL_ABORT).items():
    os.environ[k] = str(v)

import torch
cap = torch.cuda.get_device_capability(0)
print(f"\n{torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]} | "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f}GB | torch {torch.__version__}")
print(f"steps={TOTAL_STEPS} gen_len={MAX_GEN_LEN} peak_lr={PEAK_LR or 'cfg'} "
      f"hra_lr={HRA_LR or 'cfg'} tag={RUN_TAG or '(none)'} kl_abort={KL_ABORT or 'off'}")

In [ ]:
# ── 2. Prompt set: PULL the validated file, never rebuild ────────────
# Rebuilding is how empty-gold rows got in: the old builder guarded
# `gold is not None` but not `str(gold).strip()`. Such rows hit the
# `no_ground_truth` tier — +0.20 format-only — so every rollout on them scores
# identically whatever it answers, training format with NO correctness signal.
# colab_grpo.py now hard-fails on them, so a rebuilt file blocks the launch.
import json, os, shutil

from huggingface_hub import hf_hub_download

OUT = "/content/grpo_prompts.jsonl"
if not os.path.exists(OUT):
    shutil.copy2(hf_hub_download("HallD/osrt-v6-ckpt", "data/grpo_prompts.jsonl",
                                 repo_type="model"), OUT)

rows = [json.loads(l) for l in open(OUT)]
bad = [i for i, r in enumerate(rows)
       if not str(r.get("answer", "")).strip() or not str(r.get("question", "")).strip()]
print(f"{len(rows)} prompts, {len(bad)} invalid gold")
assert not bad, f"invalid gold at lines {[i+1 for i in bad[:5]]} — re-pull"

In [ ]:
# ── 3. Sanity: 3 steps. Uses the SAME budget as the run ──────────────
# A sanity check at a different generation budget cannot predict the run's
# truncation, which is the main thing worth reading from it.
#
# READ THE GENERATIONS. Every real defect this project found came from rollout
# text, not from scalars: a missing system prompt (model emitted no <|think|>,
# reward -0.984), near-miss reward hacking, and traces that compute one value
# and answer another.
!cd /content/osrt && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/colab_grpo.py \
    --ckpt-dir /content/ckpt --prompts /content/grpo_prompts.jsonl \
    --tokenizer /content/osrt/v6_tokenizer_export \
    --hf-repo HallD/osrt-v6-ckpt \
    --total-steps 3 --num-prompts 4 --ckpt-interval 999 --no-wandb \
    --max-gen-len $MAX_GEN_LEN --peak-lr $PEAK_LR --hra-lr $HRA_LR

### Reading the sanity output

**Startup lines that must be right**

- `fresh start: no local step ckpt, so the SFT-v4 soup is the base`
- `lr: peak … hra … ratio …x | prefix …` — **this is how you confirm the LR
  knobs took.** Defaults print `1.500e-06 / 1.500e-05 / 10.0x | grpo_v6b`.
- `weight EMA: decay 0.99, 229 tensors` — 229 is the FULL state_dict. Parameters
  alone would silently pair averaged weights with the latest
  `router_balance_bias`, giving a hybrid model rather than an averaged policy.
- `full-parameter GRPO: 587,288,617 base + 14,155,776 HRA` — 14.1M is rank-256.
  If it says ~884,736 the adapters were built at rank 16 and the load is wrong.
- `6000 prompts … (gold validated)`

**Rollouts must look like** `<|think|>…working…<|/think|><|answer|>72<|/answer|>`
— a think block, a **single bare number** in the answer block, clean stop.

**Red flags**

- Completions starting `<|answer|>` with no think block → the system prompt is
  not reaching the model.
- `live 0/N` → every rollout in every group scored identically, so all
  advantages are zero and **nothing is learning**.
- `cap` a large fraction of N → generations are hitting `MAX_GEN_LEN`.
  Note `cap` and `noclose` are **different**: `noclose` counts a missing
  `<|/answer|>` tag, which a model can fix by learning to emit the tag without
  ever generating shorter.

In [ ]:
# ── 4. THE RUN — foreground, so the Colab session stays alive ────────
# FOREGROUND, not nohup. A detached launch returns instantly, the notebook then
# looks IDLE, and Colab reclaims the VM out from under the background process —
# killing the run for the very reason the nohup was meant to prevent. A
# foreground cell streaming output keeps the session active; the script prints
# with flush=True, so `tee` streams rather than buffering.
#
# Losing the cell is survivable: checkpoints push to HF every 10 steps, and
# optimizer + EMA state are kept in local sidecars, so re-running resumes
# without the ~20-step AdamW moment rebuild a cold restart would cost.
import os

# MAX_GEN_LEN is the length-ramp denominator, so rewards before and after a
# change are not comparable. Clear partial artefacts of THIS prefix so a changed
# budget cannot silently resume onto a different reward scale.
prefix = "grpo_v6b" + (f"_{os.environ['RUN_TAG']}" if os.environ.get("RUN_TAG") else "")
os.makedirs("/content/ckpt", exist_ok=True)
for f in sorted(os.listdir("/content/ckpt")):
    if f.startswith(prefix):
        print("removing partial artefact:", f)
        os.remove(f"/content/ckpt/{f}")

!cd /content/osrt && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/colab_grpo.py \
    --ckpt-dir /content/ckpt \
    --prompts /content/grpo_prompts.jsonl \
    --tokenizer /content/osrt/v6_tokenizer_export \
    --hf-repo HallD/osrt-v6-ckpt \
    --ckpt-interval 10 \
    --num-prompts 16 \
    --micro-batch 24 \
    --total-steps $TOTAL_STEPS \
    --max-gen-len $MAX_GEN_LEN \
    --peak-lr $PEAK_LR --hra-lr $HRA_LR \
    --run-tag "$RUN_TAG" --kl-abort $KL_ABORT \
    --compile 2>&1 | tee -a /content/grpo.log

In [ ]:
# ── 5. Log digest (the run streams inline above) ─────────────────────
# For skimming a long run, or after a reconnect when the cell output is gone but
# /content/grpo.log survives. `tee -a` APPENDS, so this spans resumed sessions.
!grep -E "^step|rollouts @|^  \[|saved|pushed|ABORT|Error|Traceback|CUDA out of memory" \
    /content/grpo.log | tail -40

In [ ]:
# ── 6. Held-out panel eval + the PAIRED judge ────────────────────────
# Scores the candidate AND the soup on the same 200 problems, then computes the
# paired difference with a bootstrap over QUESTIONS. Self-contained: it does not
# depend on item files stored elsewhere.
#
# WHY PAIRED. Both are scored on the SAME problems, so item difficulty is
# shared: a question neither solves contributes an identical 0 to both. Treating
# the two means as independent samples understates the uncertainty — that error
# turned a p=0.186 trend into a "significant" one earlier in this project.
# Only DISCORDANT items carry information, which is why they are printed.
CANDIDATES = ["grpo_v6b_final.pt", "grpo_v6b_ema_final.pt"]
BASELINE = "osrt_v5_sft_v4_soup_1200_1400_1600_1800.pt"
N_PROBLEMS = 200
PROBLEM_OFFSET = 0     # 0 = development panel. 200 = untouched confirmation set.

import os, random, sys
import torch
sys.path.insert(0, "/content/osrt/src")
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from osrt.model import OSRTForCausalLM
from osrt.presets import build_config
from osrt.sft_eval import run_reasoning_eval

tok = AutoTokenizer.from_pretrained("/content/osrt/v6_tokenizer_export")
assert len(tok) == 65536, f"wrong tokenizer: {len(tok)} (v6 is 65536)"
cfg = build_config(vocab_size=len(tok), real_vocab_size=len(tok),
                   bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
                   pad_token_id=tok.pad_token_id, fused_cross_entropy_chunks=8)
device = torch.device("cuda")
model = OSRTForCausalLM(cfg).to(device)

def score(name):
    path = f"/content/ckpt/{name}"
    if not os.path.exists(path):
        path = hf_hub_download("HallD/osrt-v6-ckpt", name, repo_type="model")
    ck = torch.load(path, map_location=device, weights_only=True)
    miss, unexp = model.load_state_dict(ck.get("model_state_dict", ck), strict=False)
    assert not miss and not unexp, f"{name}: {miss[:3]} {unexp[:3]}"
    lr_meta = {k: ck[k] for k in ("peak_lr", "hra_lr", "kl_coeff") if k in ck}
    del ck
    model.eval()
    if hasattr(model, "set_moe_telemetry"):
        model.set_moe_telemetry(False)
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        m = run_reasoning_eval(model, tok, device, n_problems=N_PROBLEMS,
                               max_new_tokens=512, batch_size=32,
                               repetition_penalty=1.2, return_items=True,
                               problem_offset=PROBLEM_OFFSET)
    print(f"{name:<32} acc_on {100*m['sft_eval/acc_on']:5.1f}%  "
          f"acc_off {100*m['sft_eval/acc_off']:5.1f}%  "
          f"delta {100*m['sft_eval/acc_delta_on_minus_off']:+5.1f}pp  "
          f"fmt {100*m['sft_eval/format_ok_on']:5.1f}%  "
          f"len {m['sft_eval/resp_len_on']:.0f}"
          + (f"  {lr_meta}" if lr_meta else ""), flush=True)
    return m

results = {BASELINE: score(BASELINE)}
for c in CANDIDATES:
    try:
        results[c] = score(c)
    except Exception as e:
        print(f"{c}: skipped ({type(e).__name__}: {e})")

rng = random.Random(0); REPS = 10000
base_items = results[BASELINE]["items"]
print(f"\npaired differences vs the soup, {REPS} reps over resampled QUESTIONS")
for c in CANDIDATES:
    if c not in results:
        continue
    for key in ("on", "off"):
        a, b = base_items[key], results[c]["items"][key]
        n = len(a)
        d = 100.0 * (sum(a) - sum(b)) / n
        a_only = sum(1 for i in range(n) if a[i] and not b[i])
        b_only = sum(1 for i in range(n) if b[i] and not a[i])
        reps = sorted(100.0 * sum(a[i] - b[i] for i in
                      [rng.randrange(n) for _ in range(n)]) / n for _ in range(REPS))
        lo, hi = reps[int(.025 * REPS)], reps[int(.975 * REPS)]
        n_le = sum(1 for x in reps if x <= 0); n_ge = sum(1 for x in reps if x >= 0)
        p = min(1.0, 2 * (min(n_le, n_ge) + 1) / (REPS + 1))   # (+1)/(B+1)
        print(f"  acc_{key:<3} soup - {c:<30} {d:+6.2f}pp  "
              f"CI [{lo:+.2f},{hi:+.2f}]  p={p:.3f}  discordant {a_only}/{b_only}  "
              f"{'EXCLUDES 0' if not (lo <= 0 <= hi) else 'crosses 0'}")
print("\nA CI crossing zero means 'cannot distinguish', NOT 'equal'. A 200-item")
print("panel cannot establish non-inferiority at any sensible margin — that")
print("claim needs PROBLEM_OFFSET=200 with candidates declared in advance.")

In [ ]:
# ── 7. Weight drift (DIAGNOSTIC, never a verdict) ────────────────────
# Both files are already on this VM, so no download. Separates "the update went
# somewhere" from "the update went to the base model": the HRA adapters run at
# 10x the base lr, and measured HRA:base movement was 49x, so an update can look
# healthy in aggregate while the MLP/expert weights barely move.
#
# NOT a verdict. Wave 2 moved 0.145% overall and lost 7.5pp of accuracy — tiny
# movement changes behaviour materially. The judge is paired acc_on.
A = "/content/ckpt/osrt_v5_sft_v4_soup_1200_1400_1600_1800.pt"
B = "/content/ckpt/grpo_v6b_final.pt"

import torch

def bucket(nm):
    if "adapters_a" in nm or "adapters_b" in nm: return "HRA adapters"
    if "expert" in nm or "moe" in nm.lower():    return "MoE experts"
    if "embed" in nm or "lm_head" in nm:         return "embeddings"
    if "norm" in nm:                             return "norms"
    return "attention/other"

sa = torch.load(A, map_location="cpu", weights_only=True)
sb = torch.load(B, map_location="cpu", weights_only=True)
sa, sb = sa.get("model_state_dict", sa), sb.get("model_state_dict", sb)
groups, td, ta = {}, 0.0, 0.0
for k, va in sa.items():
    vb = sb.get(k)
    if vb is None or va.shape != vb.shape or not va.is_floating_point():
        continue
    fa, fb = va.float(), vb.float()
    dn = torch.linalg.vector_norm(fb - fa).item()
    an = torch.linalg.vector_norm(fa).item()
    td += dn ** 2; ta += an ** 2
    if an > 0:
        groups.setdefault(bucket(k), []).append(dn / an)
print(f"OVERALL relative L2: {100*td**0.5/ta**0.5:.4f}% of weight norm")
for g, v in sorted(groups.items()):
    v.sort()
    print(f"  {g:<16} n={len(v):>4}  median {100*v[len(v)//2]:.4f}%  max {100*v[-1]:.4f}%")
base = groups.get("attention/other", [0]); hra = groups.get("HRA adapters", [0])
bm, hm = sorted(base)[len(base)//2], sorted(hra)[len(hra)//2]
print(f"\nHRA : base movement = {hm/bm:.1f}x   (wave 2: 49x, lr ratio 10x)")

## Judging the run

**Reward is not the judge, and neither is loss.** Measured across 28 logged
steps: `corr(reward, acc) = +0.976` — reward is accuracy restated, and accuracy
on 16 prompts has a residual sd of **8.13pp**. `corr(loss, acc) = -0.154`; the
GRPO loss is a policy-gradient objective with no target, so **loss near zero is
bad**, meaning no advantage variance and nothing to learn from.

**What to watch in the step line**

- `live N/M` — rollouts with non-zero advantage. Collapsing toward 0 means
  learning has stopped whatever reward does.
- `cap` vs `noclose` — **different failures.** `cap` is hitting `MAX_GEN_LEN`;
  `noclose` is a missing `<|/answer|>` tag, which the policy can fix by learning
  to emit the tag while generating no shorter. Conflating them made
  "truncation self-corrected" look measured when cap hits were never counted.
- `kl` — the safety gauge. At `beta=0.04` it ran ~0.0024/step; raising the LR
  multiplies that, and raising the LR while lowering beta compounds. Wave 1's
  worst was 0.33.
- **The printed rollouts.** Every real defect here was found in generation text.

**Statistical discipline**

- The paired CI, not the point estimate. 200 items give roughly ±5–7pp on a
  paired difference; a 3pp gap can be perfectly reproducible *and*
  indistinguishable from zero.
- The eval is deterministic under identical settings — the same weights
  reproduce the same number exactly. That is *repeatability*, not precision
  about population accuracy.
- **Selection optimism is not priced by any of this.** Checkpoints and soups
  chosen after seeing these 200 problems import a bias no bootstrap over the
  same 200 can measure. Final claims go on `PROBLEM_OFFSET=200`.
- **Training-seed variation is unmeasured.** One trajectory. A repeat run at
  identical settings is the cheapest way to price it.

**Sweep checkpoints at the end; do not ship the last one.** In SFT-v4 the final
checkpoint was measurably not the best, and the wave-2 soup beat its own
endpoint by 1.0pp `acc_on` with a 4pp better delta. Score `theta` **and** the
EMA shadow — and check the EMA's reported *residual weight on the base* before
crediting an early EMA win, since at decay 0.99 and 50 updates the shadow is
still 61% the starting checkpoint.